In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
Path.cwd().parents[1]

In [ ]:
# we need to modify the date column in the dynLCI output files, reasons in the "check_timex_timedate" folder
folder = Path.cwd().parents[1] / "dp-LCI_output" / "hydro_ei"/ "hydro_dpLCI_v2_reservoir_wo_updateCO2CH4"   
out_folder = folder / "dateshifted_ei_hydro"

DATE_COL = "date"
FORCE_STR_COLS = ["flow", "activity"]

In [ ]:
# an extra CA-ON was run during testing the timex
files = list(folder.glob("*.xlsx"))
print("N =", len(files))

for f in files:
    print(f.name)

### caz for 2030, datetime is correct from timex, so we only prcess for 2050 LCIs

In [ ]:
for xlsx in folder.glob("*.xlsx"):
    fname = xlsx.name
    #print(fname)

    # ----------------------------------------------
    # Determine year shift based on filename
    # ----------------------------------------------
    if "2050" in fname:
        year_shift = 20
    elif "2040" in fname:
        year_shift = 10
    else:
        continue  # skip files for 2030, correct date

    print(f"Processing {fname}  →  +{year_shift} years")

    # ----------------------------------------------
    # Load Excel
    # ----------------------------------------------
    df = pd.read_excel(xlsx)
    #print(len(df))
    
    # ----------------------------------------------
    # Ensure datetime & shift years
    # ----------------------------------------------
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="raise")
    df[DATE_COL] = df[DATE_COL] + pd.DateOffset(years=year_shift)

    # ----------------------------------------------
    # Force flow & activity to string
    # ----------------------------------------------
    for col in FORCE_STR_COLS:
        if col in df.columns:
            df[col] = df[col].astype(str)

    # ----------------------------------------------
    # Save back (overwrite)
    # ----------------------------------------------
    out = out_folder / f"{xlsx.stem}.xlsx"
    df.to_excel(out, index=False)
    